# Compute isochrones around hospitals
Zehui Yin

In [ ]:
import datetime
import geopandas as gpd
import r5py

In [ ]:
# read in hospital opportunities
hospital_opportunities = gpd.read_file("output/hospital_opportunities.geojson")
hospital_opportunities["id"] = hospital_opportunities["opportunity_id"]

In [ ]:
# build the transport network
transport_network = r5py.TransportNetwork(
    "data/cropped-ontario-260320.osm.pbf",
    [
        "data/HSR_Transit_Feed.zip",
        "data/TTC_Routes_and_Schedules_Data.zip",
    ]
)

In [ ]:
# compute isochrones
isochrones = r5py.Isochrones(
    transport_network,
    origins=hospital_opportunities,
    departure=datetime.datetime(2026, 3, 17, 8, 0, 0),
    transport_modes=[r5py.TransportMode.TRANSIT],
    isochrones=[15, 30, 45], # type: ignore
)

# convert time delta to minutes
isochrones['travel_time'] = isochrones['travel_time'].apply(lambda x: x.total_seconds() / 60)

In [ ]:
isochrones.explore(column="travel_time", cmap="YlOrRd", tiles="CartoDB.Positron")

In [ ]:
isochrones.to_file("output/isochrones.geojson", driver="GeoJSON")